# GPT-2 Native Perplexity Zero-Shot CEFR Classification

This notebook reproduces the GPT-2 native perplexity zero-shot experiment
from the `minimal-cefr` repository.

**Pipeline:**
1. Extract aggregate perplexity features from pre-trained GPT-2
2. Convert to numeric feature matrices
3. Train Logistic Regression + XGBoost classifiers on EFCAMDAT-train
4. Predict on 3 test sets (EFCAMDAT-test, CELVA-SP, KUPA-KEYS)
5. Generate ranked results summary

**Runtime:** ~20 min on T4 GPU with default limits, ~3h on CPU.

**Setup:** Go to `Runtime > Change runtime type > T4 GPU` before running.

---
## 0. Install Dependencies

In [11]:
!pip install transformers torch xgboost scikit-learn pandas tqdm -q

---
## 1. Configuration

Set row limits and other experiment parameters.
Change any limit to `None` to use the full dataset (needed for paper reproduction).

In [12]:
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
import json
import torch
import os

# ── CREATE TIMESTAMP FIRST (before config initialization) ──────────────────────
TIMESTAMP = datetime.now().strftime('%Y%m%d-%H%M%S')

@dataclass
class ExperimentConfig:
    """Centralized configuration - ALL paths and settings in one place."""

    # ── GPU & BATCHING ────────────────────────────────────────────────────────
    batch_size: int = 64
    processing_mode: str = "auto"
    long_text_strategy: str = "immediate_context"
    use_batched_extraction: bool = True
    skip_merge: bool = False

    # ── GPU OPTIMIZATION ──────────────────────────────────────────────────────
    mixed_precision: bool = False       # FP16 inference (requires CUDA)
    torch_compile: bool = False         # torch.compile optimization
    max_parallel_datasets: int = 1      # 1=sequential, >1=parallel

    # ── PARALLEL CHUNK EXTRACTION ─────────────────────────────────────────────
    num_chunks: int = 10                # Split each dataset into N chunks

    # ── ROW LIMITS (for testing) ──────────────────────────────────────────────
    limit_train: int = None
    limit_test: int = None
    limit_celva: int = None
    limit_kupa: int = None

    # ── DATASET NAMES ─────────────────────────────────────────────────────────
    datasets: dict = None
    classifiers: list = None
    cefr_col: str = "cefr_level"
    feature_dir_name: str = "gpt2_native"

    # ── LOCAL PATHS (Colab environment) ───────────────────────────────────────
    repo_dir: Path = None

    # ── GOOGLE DRIVE PATHS (persistent storage) ───────────────────────────────
    # Base checkpoint directory in Google Drive
    gdrive_checkpoints_root: Path = None

    # Base experiments directory in Google Drive
    gdrive_experiments_root: Path = None

    # Data source in Google Drive
    data_path: Path = None

    # ── MANUAL RESUME PATHS (set these to resume existing experiments) ───────
    # Set these manually to reuse existing checkpoints
    resume_checkpoint_dir: Path = None
    resume_experiment_dir: Path = None

    # Dataset-specific checkpoint paths (manual override)
    # Example: {"norm-EFCAMDAT-train": Path(".../_p/cefr-classification/checkpoints/gpt2-native-.../norm-EFCAMDAT-train-...")}
    dataset_checkpoints: dict = None

    # ── DERIVED PATHS (auto-generated or from existing) ───────────────────────
    checkpoint_base: Path = None
    experiment_base: Path = None
    perplexity_raw: Path = None
    trimmed_labels: Path = None

    def __post_init__(self):
        """Initialize all derived paths after dataclass initialization."""

        # ── DEFAULT LOCAL PATHS ───────────────────────────────────────────────
        if self.repo_dir is None:
            self.repo_dir = Path("/content/minimal-cefr")

        # ── DEFAULT GDRIVE ROOTS ──────────────────────────────────────────────
        if self.gdrive_checkpoints_root is None:
            self.gdrive_checkpoints_root = Path('/content/drive/MyDrive/_p/cefr-classification/checkpoints')

        if self.gdrive_experiments_root is None:
            self.gdrive_experiments_root = Path('/content/drive/MyDrive/_p/cefr-classification/experiments')

        if self.data_path is None:
            self.data_path = Path('/content/drive/MyDrive/phd-experimental-data/cefr-classification/data/splits')

        # ── CHECKPOINT DIRECTORY: USE MANUAL PATH OR CREATE NEW ───────────────
        if self.resume_checkpoint_dir is not None:
            # Use manually specified checkpoint directory
            self.checkpoint_base = self.resume_checkpoint_dir
            print(f"\n✅ RESUMING CHECKPOINT (manual): {self.checkpoint_base}")
        else:
            # Create new checkpoint directory with timestamp
            self.checkpoint_base = self.gdrive_checkpoints_root / f'gpt2-native-{TIMESTAMP}'
            print(f"\n✅ CREATING NEW CHECKPOINT: {self.checkpoint_base}")

        # ── EXPERIMENT DIRECTORY: USE MANUAL PATH OR CREATE NEW ────────────────
        if self.resume_experiment_dir is not None:
            # Use manually specified experiment directory
            self.experiment_base = self.resume_experiment_dir
            print(f"✅ RESUMING EXISTING EXPERIMENT (manual): {self.experiment_base}")
        else:
            # Try to find corresponding experiment directory with same timestamp
            checkpoint_name = self.checkpoint_base.name
            if checkpoint_name.startswith('gpt2-native-'):
                ts = checkpoint_name.replace('gpt2-native-', '')
                experiment_path = self.gdrive_experiments_root / f'gpt2-native-{ts}'

                if experiment_path.exists():
                    # Found corresponding experiment directory
                    self.experiment_base = experiment_path
                    print(f"✅ RESUMING EXISTING EXPERIMENT: {self.experiment_base}")
                else:
                    # Create new experiment directory with same timestamp
                    self.experiment_base = experiment_path
                    print(f"✅ CREATING NEW EXPERIMENT: {self.experiment_base}")
            else:
                # Fallback: create new experiment directory with current timestamp
                self.experiment_base = self.gdrive_experiments_root / f'gpt2-native-{TIMESTAMP}'
                print(f"✅ CREATING NEW EXPERIMENT: {self.experiment_base}")

        self.perplexity_raw = self.experiment_base / "perplexity-raw"
        self.trimmed_labels = self.experiment_base / "trimmed-labels"

        # ── DEFAULT DATASET & CLASSIFIER CONFIGS ──────────────────────────────
        if self.datasets is None:
            self.datasets = {
                "norm-EFCAMDAT-train": "train",
                "norm-EFCAMDAT-test": "test",
                "norm-CELVA-SP": "test",
                "norm-KUPA-KEYS": "test",
            }

        if self.classifiers is None:
            self.classifiers = ["logistic", "xgboost"]

        if self.dataset_checkpoints is None:
            self.dataset_checkpoints = {}

    @property
    def limits(self) -> dict:
        """Get limits as dictionary for each dataset."""
        return {
            "norm-EFCAMDAT-train": self.limit_train,
            "norm-EFCAMDAT-test": self.limit_test,
            "norm-CELVA-SP": self.limit_celva,
            "norm-KUPA-KEYS": self.limit_kupa,
        }

    def get_checkpoint_dir(self, dataset_name: str) -> Path:
        """Get checkpoint directory for specific dataset.

        Returns manually specified path if available, otherwise creates new path.
        NO auto-discovery - manual specification only.
        """
        # If manually specified, use it
        if dataset_name in self.dataset_checkpoints:
            return self.dataset_checkpoints[dataset_name]

        # Otherwise, create new with current timestamp
        return self.checkpoint_base / f"{dataset_name}-{TIMESTAMP}"

    def get_experiment_dir(self, dataset_name: str) -> Path:
        """Get experiment subdirectory for specific dataset."""
        return self.experiment_base / dataset_name

    def get_features_dir(self, dataset_name: str) -> Path:
        """Get features directory for specific dataset."""
        return self.experiment_base / "features" / self.feature_dir_name / dataset_name

    def to_dict(self) -> dict:
        """Convert config to dictionary for logging."""
        return {
            'batch_size': self.batch_size,
            'processing_mode': self.processing_mode,
            'long_text_strategy': self.long_text_strategy,
            'use_batched_extraction': self.use_batched_extraction,
            'skip_merge': self.skip_merge,
            'mixed_precision': self.mixed_precision,
            'torch_compile': self.torch_compile,
            'max_parallel_datasets': self.max_parallel_datasets,
            'num_chunks': self.num_chunks,
            'checkpoint_base': str(self.checkpoint_base),
            'experiment_base': str(self.experiment_base),
            'data_path': str(self.data_path),
            'limits': self.limits,
        }

# ── INITIALIZE CONFIG ──────────────────────────────────────────────────────────
# SIMPLE DEFAULT: Create NEW checkpoint and experiment
#config = ExperimentConfig(
#    batch_size=64,
#    processing_mode="auto",
#    long_text_strategy="immediate_context",
#    use_batched_extraction=True,
#    skip_merge=False,
#)

# ADVANCED: To RESUME existing checkpoint, uncomment and edit below:
config = ExperimentConfig(
     batch_size=64,
     processing_mode="auto",
     long_text_strategy="immediate_context",
     use_batched_extraction=True,
     skip_merge=False,
     num_chunks=10,
     max_parallel_datasets=10,
     resume_checkpoint_dir=Path('/content/drive/MyDrive/_p/cefr-classification/checkpoints/gpt2-native-20260313-192642'),
     resume_experiment_dir=Path('/content/drive/MyDrive/_p/cefr-classification/experiments/gpt2-native-20260313-192642'),
     dataset_checkpoints={
         "norm-EFCAMDAT-train": Path('/content/drive/MyDrive/_p/cefr-classification/checkpoints/gpt2-native-20260313-192642/norm-EFCAMDAT-train-20260313-192642'),
         "norm-EFCAMDAT-test": Path('/content/drive/MyDrive/_p/cefr-classification/checkpoints/gpt2-native-20260313-192642/norm-EFCAMDAT-test-20260313-192642'),
         "norm-CELVA-SP": Path('/content/drive/MyDrive/_p/cefr-classification/checkpoints/gpt2-native-20260313-192642/norm-CELVA-SP-20260313-192642'),
         "norm-KUPA-KEYS": Path('/content/drive/MyDrive/_p/cefr-classification/checkpoints/gpt2-native-20260313-192642/norm-KUPA-KEYS-20260313-192642'),
     }
 )

# ── PRINT CONFIGURATION ───────────────────────────────────────────────────────
print("\n" + "="*70)
print("EXPERIMENT CONFIGURATION")
print("="*70)
for key, value in config.to_dict().items():
    if isinstance(value, dict):
        print(f"  {key}:")
        for k, v in value.items():
            print(f"    {k}: {v}")
    else:
        print(f"  {key}: {value}")
print("="*70)
print(f"\n✅ Session ID (Timestamp): {TIMESTAMP}")
print(f"ℹ️  Checkpoint Base: {config.checkpoint_base}")
print(f"ℹ️  Experiment Base: {config.experiment_base}")
print(f"ℹ️  Data Path: {config.data_path}")


✅ RESUMING CHECKPOINT (manual): /content/drive/MyDrive/_p/cefr-classification/checkpoints/gpt2-native-20260313-192642
✅ RESUMING EXISTING EXPERIMENT (manual): /content/drive/MyDrive/_p/cefr-classification/experiments/gpt2-native-20260313-192642

EXPERIMENT CONFIGURATION
  batch_size: 64
  processing_mode: auto
  long_text_strategy: immediate_context
  use_batched_extraction: True
  skip_merge: False
  mixed_precision: False
  torch_compile: False
  max_parallel_datasets: 10
  num_chunks: 10
  checkpoint_base: /content/drive/MyDrive/_p/cefr-classification/checkpoints/gpt2-native-20260313-192642
  experiment_base: /content/drive/MyDrive/_p/cefr-classification/experiments/gpt2-native-20260313-192642
  data_path: /content/drive/MyDrive/phd-experimental-data/cefr-classification/data/splits
  limits:
    norm-EFCAMDAT-train: None
    norm-EFCAMDAT-test: None
    norm-CELVA-SP: None
    norm-KUPA-KEYS: None

✅ Session ID (Timestamp): 20260314-152039
ℹ️  Checkpoint Base: /content/drive/MyDriv

---
## 2. Data Acquisition

Run **ONE** of the two options below (A or B), then continue.

### Option A: Google Drive

Use this if your data CSVs are in Google Drive.
Expected location: `MyDrive/phd-experimental-data/cefr-classification/data/splits/` containing:
- `norm-EFCAMDAT-train.csv`
- `norm-EFCAMDAT-test.csv`
- `norm-CELVA-SP.csv`
- `norm-KUPA-KEYS.csv`

In [13]:
# ── OPTION A: Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Verify data exists in the configured path
print(f"DATA_PATH = {config.data_path}")
print(f"Checking files...")

if not config.data_path.exists():
    raise FileNotFoundError(f"Data path not found: {config.data_path}")
else:
    print(f"✅ Data path exists: {config.data_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_PATH = /content/drive/MyDrive/phd-experimental-data/cefr-classification/data/splits
Checking files...
✅ Data path exists: /content/drive/MyDrive/phd-experimental-data/cefr-classification/data/splits


### Option B: Download via URL

Use this if you have a direct download link to a .zip file
containing the data CSVs.

In [ ]:
# ── OPTION B: wget + unzip ───────────────────────────────────────────
# Replace the URL below with your actual download link
DATA_URL = "YOUR_URL_HERE"  # <-- paste your .zip URL here

!wget -q --show-progress -O /content/cefr-data.zip "{DATA_URL}"
!unzip -q -o /content/cefr-data.zip -d /content/data

# WARNING: This option changes DATA_PATH from config!
# If you use this option, you must also update config.data_path manually:
# (Uncomment and update the line below if needed)
# DATA_PATH = Path('/content/data/splits')

# For now, verify the downloaded data exists
print(f"Downloaded data to: /content/data")
print(f"Current DATA_PATH (from config): {DATA_PATH}")
print(f"⚠️  If data is in a different location, update DATA_PATH and re-run cell 11")

### Verify Data

Check that all required files are present.

In [14]:
import pandas as pd

assert config.data_path is not None, "Run Option A or Option B above first!"

missing = []
for name in config.datasets:
    f = config.data_path / f"{name}.csv"
    if f.exists():
        n = len(pd.read_csv(f))
        print(f"  OK  {name}.csv ({n:,} rows)")
    else:
        print(f"  MISSING  {f}")
        missing.append(name)

if missing:
    raise FileNotFoundError(
        f"Missing files: {missing}. Check DATA_PATH or re-run data acquisition."
    )

  OK  norm-EFCAMDAT-train.csv (79,998 rows)
  OK  norm-EFCAMDAT-test.csv (20,002 rows)
  OK  norm-CELVA-SP.csv (1,742 rows)
  OK  norm-KUPA-KEYS.csv (1,006 rows)


---
## 3. Get Repository Code

Clone the `minimal-cefr` repository to get the `src/` modules.

If the repo is private, use **Alternative** below instead.

In [15]:
# Clone the repository (change URL to your fork if needed)
!git clone https://github.com/berstearns/minimal-cefr.git /content/minimal-cefr 2>/dev/null || echo "Repository already cloned."

%cd /content/minimal-cefr

# Verify src/ exists
!ls src/*.py | head -5

Repository already cloned.
/content/minimal-cefr
src/batch_processor.py
src/concat_features.py
src/config.py
src/domain.py
src/dry_run.py


**Alternative:** If you don't have a public repo, upload a zip of the `src/` directory:

```python
from google.colab import files
uploaded = files.upload()  # upload src.zip
!mkdir -p /content/minimal-cefr
!unzip -q -o src.zip -d /content/minimal-cefr
%cd /content/minimal-cefr
```

---
## 4. Experiment Setup

Create directory structure, copy data, detect GPU.

In [16]:
import shutil

# ── Device detection ──────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  WARNING: No GPU detected. Perplexity extraction will be slow.")
    print("  Go to Runtime > Change runtime type > T4 GPU")

# ── Create experiment directories ─────────────────────────────────────
dirs_to_create = [
    config.experiment_base / "ml-training-data",
    config.experiment_base / "ml-test-data",
    config.experiment_base / "features",
    config.experiment_base / "feature-models",
    config.experiment_base / "results",
    config.perplexity_raw,
    config.trimmed_labels,
]

for d in dirs_to_create:
    d.mkdir(parents=True, exist_ok=True)

# ── Copy data into experiment structure ───────────────────────────────
for name, role in config.datasets.items():
    src = config.data_path / f"{name}.csv"
    if role == "train":
        dest = config.experiment_base / "ml-training-data" / f"{name}.csv"
    else:
        dest = config.experiment_base / "ml-test-data" / f"{name}.csv"

    if not dest.exists():
        shutil.copy2(src, dest)
        print(f"  Copied {name}.csv -> {dest.parent.name}/")
    else:
        print(f"  Already exists: {dest.name}")

print("\n✅ Setup complete.")

Device: cuda
  GPU: Tesla T4
  Memory: 15.6 GB
  Already exists: norm-EFCAMDAT-train.csv
  Already exists: norm-EFCAMDAT-test.csv
  Already exists: norm-CELVA-SP.csv
  Already exists: norm-KUPA-KEYS.csv

✅ Setup complete.


In [ ]:
### Smart Default: Auto-Resume from Checkpoint

**How it works:**
1. Run the notebook normally
2. If interrupted → Colab session ends
3. Open the notebook again and run it → **Automatically resumes from checkpoint!**
4. No configuration needed - it just works!

This is the perfect workflow for Google Drive-backed experiments:
- Checkpoints persist in Google Drive
- Just keep re-running the notebook
- It picks up exactly where it stopped

**Example workflow:**

**Session 1:** `PROCESSING_MODE = "auto"` (default)
```
✓ Process dataset 1 (short texts): 2 min
✓ Process dataset 1 (long texts): 45 min
→ INTERRUPTED after 47 minutes
Checkpoint saved to Google Drive
```

**Session 2:** Same notebook, re-run it
```
Checking checkpoint...
  Checkpoint found! Auto-resuming...
    Short texts: 3500/3500 batches already done → Skipping
    Long texts: 18000/24000 batches
  Mode: RESUME from checkpoint
✓ Continue from batch 18000...
✓ Finish remaining 6000 long batches (15 more minutes)
✓ Move to next datasets
```

**No configuration needed!**

---

### ADVANCED: Processing Mode Overrides

If you need special behavior, change `PROCESSING_MODE`:

| Mode | Behavior | Use Case |
|------|----------|----------|
| `"auto"` (DEFAULT) | Resume if checkpoint exists, else start fresh | Normal usage - just works! |
| `"resume"` | Force resume (fail if no checkpoint) | Explicitly resume |
| `"all"` | Force full processing (ignore checkpoint) | Re-process everything |
| `"short_only"` | Process only short texts | Testing, or spread across days |
| `"long_only"` | Process only long texts | After short texts are done |

**Most of the time, just leave `PROCESSING_MODE = "auto"`**


### Quick Reference: Processing Modes

**How to process datasets incrementally (slowly over time):**

1. **First run (everything):**
   ```
   PROCESSING_MODE = "all"
   SKIP_MERGE = False
   ```
   → Processes short texts, long texts, and merges all results

2. **Process short texts only, long texts later:**
   ```
   PROCESSING_MODE = "short_only"
   SKIP_MERGE = True
   ```
   → Run this first, then come back later for long texts

3. **Process only long texts (after short are done):**
   ```
   PROCESSING_MODE = "long_only"
   SKIP_MERGE = True
   ```
   → Run after short texts are done

4. **Resume if interrupted:**
   ```
   PROCESSING_MODE = "resume"
   SKIP_MERGE = False
   ```
   → Automatically continues from last checkpoint

5. **Manually merge batches later:**
   ```
   SKIP_MERGE = True  # On initial run
   ```
   → Then run cell 5a (Manual Batch Merge) when ready

**Processing times:**
- Short texts (56k): ~2 min on T4 GPU
- Long texts (24k): ~30-60 min on T4 GPU (depending on text lengths)
- Total: ~45 min to 1 hour on T4 GPU

**GPU memory usage:**
- BATCH_SIZE=16: ~12-14 GB on T4
- BATCH_SIZE=8: ~7-8 GB (for smaller GPUs)

checkpoint_dir

In [ ]:
checkpoint_dir


PosixPath('/content/drive/MyDrive/phd-experimental-data/cefr-classification/checkpoints/gpt2-native-20260313-192642/norm-EFCAMDAT-train-20260313-192642')

In [ ]:
%%time
import subprocess
import sys
import pandas as pd

# Verify setup
if not config.repo_dir.exists():
    raise RuntimeError(
        f"Repository not found at {config.repo_dir}. "
        "Run cell 13 (Get Repository Code) first!"
    )

if not config.data_path.exists():
    raise RuntimeError(
        f"Data not found at {config.data_path}."
        "Run cell 7 or 8 (Data Acquisition) first!"
    )

# ── BATCHED EXTRACTION WITH AUTO-RESUME ──────────────────────────────
if config.use_batched_extraction:
    for name in config.datasets:
        limit = config.limits.get(name)
        src_csv = config.data_path / f"{name}.csv"
        checkpoint_dir = config.get_checkpoint_dir(name)
        raw_out = config.perplexity_raw / f"{name}.csv"

        print(f"\n{'='*70}")
        print(f"DATASET: {name}")
        print(f"{'='*70}")
        print(f"  Limit: {limit if limit else 'FULL'}")
        print(f"  Checkpoint Dir: {checkpoint_dir}")
        print(f"  Checkpoint Exists: {checkpoint_dir.exists()}")

        # Check if already done
        if raw_out.exists():
            print(f"  ✓ Perplexity already extracted: {raw_out.name}")
        else:
            # Build extraction command
            cmd = [
                sys.executable, "-m", "src.extract_perplexity_features_batched",
                "-i", str(src_csv),
                "--text-column", "text",
                "-m", "gpt2",
                "-d", DEVICE,
                "--aggregate-only",
                "--batch-size", str(config.batch_size),
                "--checkpoint-dir", str(checkpoint_dir),
                "--long-text-strategy", config.long_text_strategy,
                "-f", "csv",
                "-o", str(raw_out),
            ]

            # ── SMART AUTO-RESUME LOGIC ──────────────────────────────────────
            should_resume = False

            if checkpoint_dir.exists():
                checkpoint_metadata = checkpoint_dir / "metadata.json"

                if checkpoint_metadata.exists():
                    # Has metadata - check if anything was completed
                    with open(checkpoint_metadata) as f:
                        checkpoint_state = json.load(f)

                    # FIX: Use num_completed_batches (actually updated by batch_processor)
                    # NOT completed_batches (initialized to 0, never updated in old code)
                    short_done = checkpoint_state.get("short_texts", {}).get("num_completed_batches",
                                 checkpoint_state.get("short_texts", {}).get("completed_batches", 0))
                    long_done = checkpoint_state.get("long_texts", {}).get("num_completed_batches",
                                checkpoint_state.get("long_texts", {}).get("completed_batches", 0))

                    print(f"  Checkpoint found with metadata")
                    print(f"    Short texts: {short_done} batches done")
                    print(f"    Long texts: {long_done} batches done")

                    short_total = checkpoint_state.get("short_texts", {}).get("total_batches", 0)
                    long_total = checkpoint_state.get("long_texts", {}).get("total_batches", 0)

                    # Only resume if something was actually completed
                    if short_done > 0 or long_done > 0:
                        print(f"  ✓ RESUMING from checkpoint (has progress)")
                        should_resume = True

                        if short_done >= short_total and short_total > 0:
                            print(f"    → Short texts 100% done, skipping")
                            cmd.append("--skip-short-texts")
                    else:
                        print(f"  ✗ Checkpoint exists but empty (0 batches). Starting fresh.")
                        should_resume = False
                else:
                    # Checkpoint dir exists but no metadata - assume it's empty/corrupted
                    print(f"  Checkpoint exists but no metadata. Starting fresh.")
                    should_resume = False
            else:
                print(f"  ✗ Checkpoint: Starting fresh (directory does not exist)")
                should_resume = False

            # Add resume flag only if we actually have progress
            if should_resume:
                cmd.append("--resume")
                print(f"  Mode: RESUME from checkpoint")
            else:
                print(f"  Mode: START FROM BEGINNING")

            # Handle PROCESSING_MODE overrides
            if config.processing_mode == "short_only":
                cmd.append("--skip-long-texts")
                print(f"  Override: SHORT TEXTS ONLY")
            elif config.processing_mode == "long_only":
                cmd.append("--skip-short-texts")
                print(f"  Override: LONG TEXTS ONLY")
            elif config.processing_mode == "all":
                if "--resume" in cmd:
                    cmd.remove("--resume")
                print(f"  Override: FORCE FULL PROCESSING (ignoring checkpoint)")

            if config.skip_merge:
                cmd.append("--no-merge")
                print(f"  Merge: DISABLED (batches kept separate)")

            if limit is not None:
                cmd.extend(["--limit", str(limit)])

            print(f"  Strategy: {config.long_text_strategy}")
            print(f"  Batch size: {config.batch_size}")
            print(f"\n  Starting extraction...")

            result = subprocess.run(
                cmd,
                cwd=str(config.repo_dir),
                capture_output=True,
                text=True,
                timeout=7200
            )

            if result.returncode != 0:
                print(f"  ❌ FAILED")
                print(f"  STDERR: {result.stderr[-2000:]}")
                raise RuntimeError(f"Batched extraction failed for {name}")

            for line in result.stdout.strip().split('\n')[-8:]:
                if line.strip():
                    print(f"  {line}")

        # Step 2: Convert to features
        feat_dir = config.get_features_dir(name)
        feat_dir.mkdir(parents=True, exist_ok=True)
        feat_out = feat_dir / "features_dense.csv"

        if not feat_out.exists():
            print(f"\n  Converting to feature matrix...")
            df = pd.read_csv(raw_out)
            drop_cols = [c for c in ["text", "model"] if c in df.columns]
            df_numeric = df.drop(columns=drop_cols)

            for col in df_numeric.columns:
                df_numeric[col] = pd.to_numeric(df_numeric[col], errors="coerce")

            df_numeric = df_numeric.fillna(0.0)
            df_numeric.to_csv(feat_out, index=False)

            fn_file = feat_dir / "feature_names.csv"
            pd.DataFrame({"feature_name": df_numeric.columns}).to_csv(fn_file, index=False)

            print(f"  ✓ Features: {len(df_numeric):,} rows × {len(df_numeric.columns)} cols")
            print(f"    Saved: {feat_out}")
        else:
            print(f"  ✓ Features already converted: {feat_out.name}")

else:
    print("\n⚠️  Using standard (non-batched) extraction - will be slower")
    print("   Set use_batched_extraction = True for 10-15x speedup\n")

    for name in config.datasets:
        limit = config.limits.get(name)
        src_csv = config.data_path / f"{name}.csv"
        raw_out = config.perplexity_raw / f"{name}.csv"

        print(f"\n{'='*70}")
        print(f"  {name} (limit={limit})")
        print(f"{'='*70}")

        if not raw_out.exists():
            cmd = [
                sys.executable, "-m", "src.extract_perplexity_features",
                "-i", str(src_csv),
                "--text-column", "text",
                "-m", "gpt2",
                "-d", DEVICE,
                "--aggregate-only",
                "-f", "csv",
                "-o", str(raw_out),
            ]
            if limit is not None:
                cmd.extend(["--limit", str(limit)])

            print(f"  Extracting perplexity...")
            result = subprocess.run(cmd, cwd=str(config.repo_dir), capture_output=True, text=True, timeout=7200)
            if result.returncode != 0:
                print(f"  STDERR: {result.stderr[-2000:]}")
                raise RuntimeError(f"Perplexity extraction failed for {name}")

            for line in result.stdout.strip().split('\n')[-5:]:
                print(f"  {line}")
        else:
            print(f"  ✓ Perplexity already extracted.")

        feat_dir = config.get_features_dir(name)
        feat_dir.mkdir(parents=True, exist_ok=True)
        feat_out = feat_dir / "features_dense.csv"

        if not feat_out.exists():
            df = pd.read_csv(raw_out)
            drop_cols = [c for c in ["text", "model"] if c in df.columns]
            df_numeric = df.drop(columns=drop_cols)

            for col in df_numeric.columns:
                df_numeric[col] = pd.to_numeric(df_numeric[col], errors="coerce")

            df_numeric = df_numeric.fillna(0.0)
            df_numeric.to_csv(feat_out, index=False)

            fn_file = feat_dir / "feature_names.csv"
            pd.DataFrame({"feature_name": df_numeric.columns}).to_csv(fn_file, index=False)

            print(f"  ✓ Features: {len(df_numeric)} rows x {len(df_numeric.columns)} cols")
        else:
            print(f"  ✓ Features already converted.")

print("\n" + "="*70)
print("Step 1+2 complete.")
print("="*70)

### Alternative: GPU-Optimized Extraction (cell 21b)

Run **either** cell 21 (standard) **or** cell 21b (GPU-optimized) - not both.

This cell adds:
- **Mixed precision (FP16):** `mixed_precision=True` in config - reduces memory, ~1.3x faster
- **torch.compile:** `torch_compile=True` in config - JIT compilation, ~1.2x faster (first batch slow)
- **Parallel datasets:** `max_parallel_datasets=2` in config - run 2 datasets simultaneously

All flags default to disabled = same behavior as cell 21.

In [ ]:
%%time
import subprocess
import sys
import time
import pandas as pd

# ── VERIFY SETUP ─────────────────────────────────────────────────────────────
if not config.repo_dir.exists():
    raise RuntimeError(
        f"Repository not found at {config.repo_dir}. "
        "Run cell 13 (Get Repository Code) first!"
    )

if not config.data_path.exists():
    raise RuntimeError(
        f"Data not found at {config.data_path}. "
        "Run cell 7 or 8 (Data Acquisition) first!"
    )

print("=" * 70)
print("GPU-OPTIMIZED EXTRACTION (cell 21b)")
print("=" * 70)
print(f"  Mixed precision (FP16): {'ENABLED' if config.mixed_precision else 'disabled'}")
print(f"  torch.compile:          {'ENABLED' if config.torch_compile else 'disabled'}")
print(f"  Parallel datasets:      {config.max_parallel_datasets}")
print(f"  Batch size:             {config.batch_size}")
print(f"  Device:                 {DEVICE}")
print("=" * 70)


def build_extraction_cmd(name, config, DEVICE):
    """Build subprocess command for a single dataset extraction."""
    limit = config.limits.get(name)
    src_csv = config.data_path / f"{name}.csv"
    checkpoint_dir = config.get_checkpoint_dir(name)
    raw_out = config.perplexity_raw / f"{name}.csv"

    # Already done?
    if raw_out.exists():
        return None, raw_out, f"Already extracted: {raw_out.name}"

    cmd = [
        sys.executable, "-m", "src.extract_perplexity_features_batched",
        "-i", str(src_csv),
        "--text-column", "text",
        "-m", "gpt2",
        "-d", DEVICE,
        "--aggregate-only",
        "--batch-size", str(config.batch_size),
        "--checkpoint-dir", str(checkpoint_dir),
        "--long-text-strategy", config.long_text_strategy,
        "-f", "csv",
        "-o", str(raw_out),
    ]

    # GPU optimization flags
    if config.mixed_precision:
        cmd.append("--mixed-precision")
    if config.torch_compile:
        cmd.append("--torch-compile")

    # Smart auto-resume logic
    should_resume = False
    if checkpoint_dir.exists():
        checkpoint_metadata = checkpoint_dir / "metadata.json"
        if checkpoint_metadata.exists():
            with open(checkpoint_metadata) as f:
                checkpoint_state = json.load(f)

            short_done = checkpoint_state.get("short_texts", {}).get(
                "num_completed_batches",
                checkpoint_state.get("short_texts", {}).get("completed_batches", 0))
            long_done = checkpoint_state.get("long_texts", {}).get(
                "num_completed_batches",
                checkpoint_state.get("long_texts", {}).get("completed_batches", 0))
            short_total = checkpoint_state.get("short_texts", {}).get("total_batches", 0)

            if short_done > 0 or long_done > 0:
                should_resume = True
                if short_done >= short_total and short_total > 0:
                    cmd.append("--skip-short-texts")

    if should_resume:
        cmd.append("--resume")

    # Processing mode overrides
    if config.processing_mode == "short_only":
        cmd.append("--skip-long-texts")
    elif config.processing_mode == "long_only":
        cmd.append("--skip-short-texts")
    elif config.processing_mode == "all":
        if "--resume" in cmd:
            cmd.remove("--resume")

    if config.skip_merge:
        cmd.append("--no-merge")

    if limit is not None:
        cmd.extend(["--limit", str(limit)])

    return cmd, raw_out, None


def convert_to_features(name, raw_out, config):
    """Convert raw perplexity CSV to feature matrix."""
    feat_dir = config.get_features_dir(name)
    feat_dir.mkdir(parents=True, exist_ok=True)
    feat_out = feat_dir / "features_dense.csv"

    if feat_out.exists():
        print(f"  Features already converted: {feat_out.name}")
        return

    print(f"  Converting to feature matrix...")
    df = pd.read_csv(raw_out)
    drop_cols = [c for c in ["text", "model"] if c in df.columns]
    df_numeric = df.drop(columns=drop_cols)

    for col in df_numeric.columns:
        df_numeric[col] = pd.to_numeric(df_numeric[col], errors="coerce")

    df_numeric = df_numeric.fillna(0.0)
    df_numeric.to_csv(feat_out, index=False)

    fn_file = feat_dir / "feature_names.csv"
    pd.DataFrame({"feature_name": df_numeric.columns}).to_csv(fn_file, index=False)

    print(f"  Features: {len(df_numeric):,} rows x {len(df_numeric.columns)} cols -> {feat_out}")


# ── MAIN LOGIC: SEQUENTIAL vs PARALLEL ───────────────────────────────────────
dataset_names = list(config.datasets.keys())

if config.max_parallel_datasets <= 1:
    # ── SEQUENTIAL MODE (same as cell 21, with GPU flags) ────────────────
    for name in dataset_names:
        print(f"\n{'='*70}")
        print(f"DATASET: {name}")
        print(f"{'='*70}")

        cmd, raw_out, skip_msg = build_extraction_cmd(name, config, DEVICE)

        if skip_msg:
            print(f"  {skip_msg}")
        else:
            print(f"  Starting extraction...")
            result = subprocess.run(
                cmd,
                cwd=str(config.repo_dir),
                capture_output=True,
                text=True,
                timeout=7200
            )

            if result.returncode != 0:
                print(f"  FAILED")
                print(f"  STDERR: {result.stderr[-2000:]}")
                raise RuntimeError(f"Extraction failed for {name}")

            for line in result.stdout.strip().split('\n')[-8:]:
                if line.strip():
                    print(f"  {line}")

        # Clear GPU memory between datasets
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

        convert_to_features(name, raw_out, config)

else:
    # ── PARALLEL MODE ────────────────────────────────────────────────────
    print(f"\nLaunching up to {config.max_parallel_datasets} datasets in parallel...")

    dataset_queue = dataset_names.copy()
    active_processes = {}
    completed = {}
    start_time = time.time()

    while dataset_queue or active_processes:
        # Launch new processes while under limit
        while len(active_processes) < config.max_parallel_datasets and dataset_queue:
            name = dataset_queue.pop(0)
            cmd, raw_out, skip_msg = build_extraction_cmd(name, config, DEVICE)

            if skip_msg:
                print(f"  {name}: {skip_msg}")
                completed[name] = (True, 0, raw_out)
                continue

            try:
                p = subprocess.Popen(
                    cmd,
                    cwd=str(config.repo_dir),
                    stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE,
                    text=True
                )
                active_processes[name] = {
                    "process": p,
                    "start_time": time.time(),
                    "raw_out": raw_out,
                }
                print(f"  [{time.strftime('%H:%M:%S')}] LAUNCHED: {name} (PID: {p.pid})")
            except Exception as e:
                print(f"  {name}: Failed to launch - {e}")
                completed[name] = (False, 0, raw_out)

        # Check for completed processes
        finished = []
        for name, proc_info in active_processes.items():
            p = proc_info["process"]
            if p.poll() is not None:
                elapsed = time.time() - proc_info["start_time"]
                if p.returncode == 0:
                    print(f"  [{time.strftime('%H:%M:%S')}] DONE: {name} ({elapsed:.1f}s)")
                    completed[name] = (True, elapsed, proc_info["raw_out"])
                else:
                    _, stderr = p.communicate()
                    print(f"  [{time.strftime('%H:%M:%S')}] FAILED: {name}")
                    print(f"    Error: {stderr[-500:] if stderr else 'Unknown'}")
                    completed[name] = (False, elapsed, proc_info["raw_out"])
                finished.append(name)

        for name in finished:
            del active_processes[name]

        if active_processes:
            time.sleep(2)

    # Clear GPU memory after parallel work
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    # Summary
    total_time = time.time() - start_time
    print(f"\nParallel extraction: {total_time:.1f}s ({total_time/60:.1f}m)")
    for name in dataset_names:
        if name in completed:
            success, elapsed, _ = completed[name]
            print(f"  {'OK' if success else 'FAIL'} {name}: {elapsed:.1f}s")

    # Convert features for all successful datasets
    for name in dataset_names:
        if name in completed:
            success, _, raw_out = completed[name]
            if success and raw_out.exists():
                convert_to_features(name, raw_out, config)
            elif not success:
                raise RuntimeError(f"Extraction failed for {name}")

print("\n" + "=" * 70)
print("GPU-optimized extraction complete.")
print("=" * 70)

### Parallel Chunk Extraction (cell 21c)

Run **either** cell 21, cell 21b, **or** cell 21c — not multiple.

This cell splits each dataset into `num_chunks` (default 10) pieces and processes them
in parallel using a worker pool of `max_parallel_datasets` simultaneous subprocesses.

**Why chunks?** Cells 21/21b parallelize *across* datasets, but each dataset runs single-threaded.
This cell parallelizes *within* a single dataset by splitting the CSV into chunks.

**VRAM note:** Each subprocess loads its own GPT-2 model (~500MB). On T4 (15.6GB),
set `max_parallel_datasets=2` (safe) or `3` (tight). Do NOT run 10 simultaneously.

**Config fields used:**
- `num_chunks` — number of pieces per dataset (default 10)
- `max_parallel_datasets` — max simultaneous workers (default 1)
- `mixed_precision`, `torch_compile` — passed through to each subprocess

---

#### Resume Strategy

All chunk state is persisted in `{checkpoint_dir}/chunks/chunks_metadata.json`:

```json
{
  "total_rows": 79998,
  "num_chunks": 10,
  "status": "processing",
  "chunks": [
    {"index": 0, "status": "completed", "num_rows": 8000, ...},
    {"index": 1, "status": "in_progress", "num_rows": 8000, ...},
    {"index": 2, "status": "pending", "num_rows": 8000, ...},
    ...
  ]
}
```

**On re-run, the cell automatically:**
1. Detects `chunks_metadata.json` in the checkpoint folder
2. Scans disk to refresh each chunk's actual status:
   - `completed` → output CSV exists → **skip**
   - `in_progress` → checkpoint has partial batches → **resume with `--resume`**
   - `pending` → no progress → **start fresh**
3. Only launches workers for non-completed chunks
4. After all chunks finish, merges and marks status as `"merged"`

**To resume:** Just re-run the cell. Point `resume_checkpoint_dir` / `dataset_checkpoints`
to the same checkpoint folder from the previous session — the chunk state is all there.

---

**Output paths:**
- Chunk CSVs + outputs: `{checkpoint_dir}/chunks/chunk_XX.csv`, `chunk_XX_output.csv`
- Per-chunk checkpoints: `{checkpoint_dir}/chunks/chunk_XX_ckpt/`
- Chunk metadata: `{checkpoint_dir}/chunks/chunks_metadata.json`
- Merged output: `{perplexity_raw}/{dataset}.csv` (same as cell 21)

In [ ]:
%%time
import subprocess
import sys
import time
import math
import pandas as pd

# ── VERIFY SETUP ─────────────────────────────────────────────────────────────
if not config.repo_dir.exists():
    raise RuntimeError(
        f"Repository not found at {config.repo_dir}. "
        "Run cell 13 (Get Repository Code) first!"
    )

if not config.data_path.exists():
    raise RuntimeError(
        f"Data not found at {config.data_path}. "
        "Run cell 7 or 8 (Data Acquisition) first!"
    )

num_chunks = config.num_chunks
max_workers = config.max_parallel_datasets

print("=" * 70)
print("PARALLEL CHUNK EXTRACTION (cell 21c)")
print("=" * 70)
print(f"  Chunks per dataset:     {num_chunks}")
print(f"  Max parallel workers:   {max_workers}")
print(f"  Mixed precision (FP16): {'ENABLED' if config.mixed_precision else 'disabled'}")
print(f"  torch.compile:          {'ENABLED' if config.torch_compile else 'disabled'}")
print(f"  Batch size:             {config.batch_size}")
print(f"  Device:                 {DEVICE}")
print("=" * 70)


def load_chunks_metadata(chunks_dir):
    """Load chunks_metadata.json if it exists."""
    meta_path = chunks_dir / "chunks_metadata.json"
    if meta_path.exists():
        with open(meta_path) as f:
            return json.load(f)
    return None


def save_chunks_metadata(chunks_dir, metadata):
    """Save chunks_metadata.json atomically."""
    meta_path = chunks_dir / "chunks_metadata.json"
    tmp_path = chunks_dir / "chunks_metadata.json.tmp"
    with open(tmp_path, "w") as f:
        json.dump(metadata, f, indent=2)
    tmp_path.rename(meta_path)


def create_chunks_metadata(chunks_dir, df_full, num_chunks, src_csv):
    """Create initial chunks_metadata.json and write chunk CSVs."""
    total_rows = len(df_full)
    chunk_size = math.ceil(total_rows / num_chunks)
    actual_chunks = math.ceil(total_rows / chunk_size)

    chunks = []
    for i in range(actual_chunks):
        start = i * chunk_size
        end = min(start + chunk_size, total_rows)
        chunk_csv = chunks_dir / f"chunk_{i:02d}.csv"
        chunk_out = chunks_dir / f"chunk_{i:02d}_output.csv"
        chunk_ckpt = chunks_dir / f"chunk_{i:02d}_ckpt"

        # Write chunk CSV
        if not chunk_csv.exists():
            df_full.iloc[start:end].to_csv(chunk_csv, index=False)

        chunks.append({
            "index": i,
            "start_row": start,
            "end_row": end,
            "num_rows": end - start,
            "chunk_csv": str(chunk_csv.name),
            "chunk_output": str(chunk_out.name),
            "chunk_ckpt": str(chunk_ckpt.name),
            "status": "pending",
        })

    metadata = {
        "source_csv": str(src_csv),
        "total_rows": total_rows,
        "num_chunks": actual_chunks,
        "chunk_size": chunk_size,
        "status": "splitting",
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "chunks": chunks,
    }

    save_chunks_metadata(chunks_dir, metadata)
    return metadata


def get_chunk_status(chunks_dir, chunk_meta):
    """Determine actual status of a chunk by checking files on disk."""
    chunk_out = chunks_dir / chunk_meta["chunk_output"]
    chunk_ckpt = chunks_dir / chunk_meta["chunk_ckpt"]

    # Output exists -> completed
    if chunk_out.exists():
        return "completed"

    # Checkpoint exists with progress -> in_progress (can resume)
    if chunk_ckpt.exists():
        ckpt_metadata = chunk_ckpt / "metadata.json"
        if ckpt_metadata.exists():
            with open(ckpt_metadata) as f:
                ckpt = json.load(f)
            short_done = ckpt.get("short_texts", {}).get(
                "num_completed_batches",
                ckpt.get("short_texts", {}).get("completed_batches", 0))
            long_done = ckpt.get("long_texts", {}).get(
                "num_completed_batches",
                ckpt.get("long_texts", {}).get("completed_batches", 0))
            if short_done > 0 or long_done > 0:
                return "in_progress"

    return "pending"


def refresh_chunks_metadata(chunks_dir, metadata):
    """Re-scan chunk statuses from disk and update metadata."""
    for chunk_meta in metadata["chunks"]:
        actual = get_chunk_status(chunks_dir, chunk_meta)
        chunk_meta["status"] = actual
    save_chunks_metadata(chunks_dir, metadata)
    return metadata


def build_chunk_cmd(chunks_dir, chunk_meta, config, DEVICE):
    """Build subprocess command for a single chunk."""
    chunk_csv = chunks_dir / chunk_meta["chunk_csv"]
    chunk_out = chunks_dir / chunk_meta["chunk_output"]
    chunk_ckpt = chunks_dir / chunk_meta["chunk_ckpt"]

    cmd = [
        sys.executable, "-m", "src.extract_perplexity_features_batched",
        "-i", str(chunk_csv),
        "--text-column", "text",
        "-m", "gpt2",
        "-d", DEVICE,
        "--aggregate-only",
        "--batch-size", str(config.batch_size),
        "--checkpoint-dir", str(chunk_ckpt),
        "--long-text-strategy", config.long_text_strategy,
        "-f", "csv",
        "-o", str(chunk_out),
    ]

    if config.mixed_precision:
        cmd.append("--mixed-precision")
    if config.torch_compile:
        cmd.append("--torch-compile")

    # Resume logic: check chunk checkpoint for partial progress
    if chunk_meta["status"] == "in_progress":
        cmd.append("--resume")
        # Check if short texts are 100% done -> skip them
        ckpt_metadata = chunk_ckpt / "metadata.json"
        if ckpt_metadata.exists():
            with open(ckpt_metadata) as f:
                ckpt = json.load(f)
            short_done = ckpt.get("short_texts", {}).get(
                "num_completed_batches",
                ckpt.get("short_texts", {}).get("completed_batches", 0))
            short_total = ckpt.get("short_texts", {}).get("total_batches", 0)
            if short_done >= short_total and short_total > 0:
                cmd.append("--skip-short-texts")

    return cmd


def read_chunk_progress(chunks_dir, chunk_meta):
    """Read batch-level progress from a chunk's checkpoint metadata.json."""
    ckpt_dir = chunks_dir / chunk_meta["chunk_ckpt"]
    meta_path = ckpt_dir / "metadata.json"
    if not meta_path.exists():
        return {"short_done": 0, "short_total": 0, "long_done": 0, "long_total": 0}
    try:
        with open(meta_path) as f:
            ckpt = json.load(f)
        short = ckpt.get("short_texts", {})
        long = ckpt.get("long_texts", {})
        return {
            "short_done": short.get("num_completed_batches", short.get("completed_batches", 0)),
            "short_total": short.get("total_batches", 0),
            "long_done": long.get("num_completed_batches", long.get("completed_batches", 0)),
            "long_total": long.get("total_batches", 0),
        }
    except Exception:
        return {"short_done": 0, "short_total": 0, "long_done": 0, "long_total": 0}


def format_elapsed(seconds):
    """Format seconds as human-readable string."""
    if seconds < 60:
        return f"{seconds:.0f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}m"
    else:
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        return f"{h}h{m:02d}m"


def convert_to_features(df, feat_dir):
    """Convert raw perplexity output to feature matrix. Returns (df_numeric, feat_out)."""
    feat_dir.mkdir(parents=True, exist_ok=True)
    feat_out = feat_dir / "features_dense.csv"
    if feat_out.exists():
        return None, feat_out  # Already done

    drop_cols = [c for c in ["text", "model"] if c in df.columns]
    df_numeric = df.drop(columns=drop_cols)
    for col in df_numeric.columns:
        df_numeric[col] = pd.to_numeric(df_numeric[col], errors="coerce")
    df_numeric = df_numeric.fillna(0.0)
    df_numeric.to_csv(feat_out, index=False)
    fn_file = feat_dir / "feature_names.csv"
    pd.DataFrame({"feature_name": df_numeric.columns}).to_csv(fn_file, index=False)
    return df_numeric, feat_out


# ── MAIN LOOP: PROCESS EACH DATASET ─────────────────────────────────────────
for name in config.datasets:
    limit = config.limits.get(name)
    src_csv = config.data_path / f"{name}.csv"
    raw_out = config.perplexity_raw / f"{name}.csv"
    checkpoint_dir = config.get_checkpoint_dir(name)

    print(f"\n{'='*70}")
    print(f"DATASET: {name}")
    print(f"{'='*70}")

    # ── SKIP IF FINAL OUTPUT EXISTS ──────────────────────────────────────
    if raw_out.exists():
        print(f"  Already extracted: {raw_out.name}")
        feat_dir = config.get_features_dir(name)
        df_numeric, feat_out = convert_to_features(pd.read_csv(raw_out), feat_dir)
        if df_numeric is not None:
            print(f"  Features: {len(df_numeric):,} rows x {len(df_numeric.columns)} cols -> {feat_out}")
        else:
            print(f"  Features already converted: {feat_out.name}")
        continue

    # ── STEP 1: INITIALIZE OR RESUME CHUNKS ──────────────────────────────
    chunks_dir = checkpoint_dir / "chunks"
    chunks_dir.mkdir(parents=True, exist_ok=True)

    chunks_meta = load_chunks_metadata(chunks_dir)

    if chunks_meta is not None:
        # ── RESUME: Refresh chunk statuses from disk ─────────────────────
        print(f"  Found existing chunks_metadata.json — resuming")
        chunks_meta = refresh_chunks_metadata(chunks_dir, chunks_meta)
        total_rows = chunks_meta["total_rows"]

        completed = [c for c in chunks_meta["chunks"] if c["status"] == "completed"]
        in_progress = [c for c in chunks_meta["chunks"] if c["status"] == "in_progress"]
        pending = [c for c in chunks_meta["chunks"] if c["status"] == "pending"]

        print(f"  Total rows: {total_rows:,}")
        print(f"  Chunks: {chunks_meta['num_chunks']} (completed={len(completed)}, "
              f"resumable={len(in_progress)}, pending={len(pending)})")

        if chunks_meta.get("status") == "merged":
            print(f"  Chunks already merged — skipping to feature conversion")
            # Re-create the merged output from chunk outputs
            merged_dfs = []
            for chunk_meta in chunks_meta["chunks"]:
                chunk_df = pd.read_csv(chunks_dir / chunk_meta["chunk_output"])
                merged_dfs.append(chunk_df)
            merged = pd.concat(merged_dfs, ignore_index=True)
            config.perplexity_raw.mkdir(parents=True, exist_ok=True)
            merged.to_csv(raw_out, index=False)
            print(f"  Re-merged: {len(merged):,} rows -> {raw_out}")
            total_rows = len(merged)
            chunks_meta["status"] = "merged"
            save_chunks_metadata(chunks_dir, chunks_meta)
            # Feature conversion
            feat_dir = config.get_features_dir(name)
            df_numeric, feat_out = convert_to_features(merged, feat_dir)
            if df_numeric is not None:
                print(f"  Features: {len(df_numeric):,} rows x {len(df_numeric.columns)} cols -> {feat_out}")
            else:
                print(f"  Features already converted: {feat_out.name}")
            continue

    else:
        # ── FRESH START: Split CSV into chunks ───────────────────────────
        print(f"  No existing chunks — creating fresh split")
        df_full = pd.read_csv(src_csv)
        if limit is not None:
            df_full = df_full.head(limit)
        total_rows = len(df_full)

        chunks_meta = create_chunks_metadata(chunks_dir, df_full, num_chunks, src_csv)
        del df_full  # Free memory

        print(f"  Total rows: {total_rows:,}")
        print(f"  Chunks: {chunks_meta['num_chunks']} x ~{chunks_meta['chunk_size']:,} rows")
        print(f"  Chunk CSVs saved to: {chunks_dir}")

        completed = []
        in_progress = []
        pending = chunks_meta["chunks"].copy()

    # ── STEP 2: WORKER POOL ──────────────────────────────────────────────
    work_queue = [c for c in chunks_meta["chunks"] if c["status"] != "completed"]

    if not work_queue:
        print(f"  All {chunks_meta['num_chunks']} chunks already completed")
    else:
        chunks_meta["status"] = "processing"
        save_chunks_metadata(chunks_dir, chunks_meta)

        print(f"  Processing {len(work_queue)} chunks with {max_workers} workers...")
        active_procs = {}
        queue = work_queue.copy()
        results = {}
        pool_start = time.time()
        last_progress_time = pool_start

        while queue or active_procs:
            # Launch workers up to max_workers
            while len(active_procs) < max_workers and queue:
                chunk_meta = queue.pop(0)
                idx = chunk_meta["index"]
                cmd = build_chunk_cmd(chunks_dir, chunk_meta, config, DEVICE)
                resume_tag = " [RESUME]" if chunk_meta["status"] == "in_progress" else ""

                try:
                    # Redirect stdout/stderr to log files to avoid pipe deadlock
                    log_out_path = chunks_dir / f"chunk_{idx:02d}.stdout.log"
                    log_err_path = chunks_dir / f"chunk_{idx:02d}.stderr.log"
                    log_out_f = open(log_out_path, "w")
                    log_err_f = open(log_err_path, "w")

                    p = subprocess.Popen(
                        cmd,
                        cwd=str(config.repo_dir),
                        stdout=log_out_f,
                        stderr=log_err_f,
                    )
                    active_procs[idx] = {
                        "process": p,
                        "start_time": time.time(),
                        "log_out": log_out_f,
                        "log_err": log_err_f,
                        "log_out_path": log_out_path,
                        "log_err_path": log_err_path,
                    }

                    # Update metadata
                    chunk_meta["status"] = "in_progress"
                    save_chunks_metadata(chunks_dir, chunks_meta)

                    print(f"    [{time.strftime('%H:%M:%S')}] LAUNCHED chunk {idx:02d}"
                          f" ({chunk_meta['num_rows']:,} rows, PID {p.pid}){resume_tag}")
                except Exception as e:
                    print(f"    chunk {idx:02d}: Failed to launch - {e}")
                    results[idx] = (False, 0)

            # Poll for completions
            finished = []
            for idx, info in active_procs.items():
                p = info["process"]
                if p.poll() is not None:
                    elapsed = time.time() - info["start_time"]
                    # Close log file handles
                    info["log_out"].close()
                    info["log_err"].close()

                    if p.returncode == 0:
                        print(f"    [{time.strftime('%H:%M:%S')}] DONE    chunk {idx:02d} ({elapsed:.1f}s)")
                        results[idx] = (True, elapsed)
                        chunks_meta["chunks"][idx]["status"] = "completed"
                        save_chunks_metadata(chunks_dir, chunks_meta)
                    else:
                        stderr_text = ""
                        try:
                            stderr_text = info["log_err_path"].read_text()
                        except Exception:
                            pass
                        print(f"    [{time.strftime('%H:%M:%S')}] FAILED  chunk {idx:02d}")
                        print(f"      Error: {stderr_text[-500:] if stderr_text else 'Unknown (check log)'}")
                        print(f"      Logs:  {info['log_out_path']}")
                        results[idx] = (False, elapsed)
                        chunks_meta["chunks"][idx]["status"] = "failed"
                        save_chunks_metadata(chunks_dir, chunks_meta)
                    finished.append(idx)

            for idx in finished:
                del active_procs[idx]

            # Periodic progress summary with batch-level detail (every 60s)
            now = time.time()
            if active_procs and (now - last_progress_time) >= 60:
                last_progress_time = now
                done_count = sum(1 for _, (ok, _) in results.items() if ok)
                total_count = len(work_queue)
                elapsed_total = now - pool_start

                # Read batch-level progress from each active chunk's checkpoint
                total_batches_done = 0
                total_batches_all = 0
                chunk_details = []
                for cidx in sorted(active_procs.keys()):
                    cmeta = chunks_meta["chunks"][cidx]
                    prog = read_chunk_progress(chunks_dir, cmeta)
                    s_done, s_total = prog["short_done"], prog["short_total"]
                    l_done, l_total = prog["long_done"], prog["long_total"]
                    c_done = s_done + l_done
                    c_total = s_total + l_total
                    total_batches_done += c_done
                    total_batches_all += c_total
                    if c_total > 0:
                        pct = c_done / c_total * 100
                        # Show which phase (short/long)
                        if s_done < s_total:
                            phase = f"S:{s_done}/{s_total}"
                        elif l_total > 0:
                            phase = f"L:{l_done}/{l_total}"
                        else:
                            phase = f"S:{s_done}/{s_total}"
                        chunk_details.append(f"c{cidx:02d}={pct:.0f}%({phase})")
                    else:
                        chunk_details.append(f"c{cidx:02d}=init")

                # Also add completed chunks to totals for ETA calc
                for cidx, (ok, _) in results.items():
                    if ok:
                        cmeta = chunks_meta["chunks"][cidx]
                        prog = read_chunk_progress(chunks_dir, cmeta)
                        total_batches_done += prog["short_done"] + prog["long_done"]
                        total_batches_all += prog["short_total"] + prog["long_total"]

                # Overall percentage and ETA
                if total_batches_all > 0:
                    overall_pct = total_batches_done / total_batches_all * 100
                    if total_batches_done > 0:
                        rate = total_batches_done / elapsed_total  # batches/sec
                        remaining = (total_batches_all - total_batches_done) / rate
                        eta_str = f"ETA ~{format_elapsed(remaining)}"
                    else:
                        eta_str = "ETA calculating..."
                else:
                    overall_pct = 0
                    eta_str = "ETA unknown"

                print(f"    [{time.strftime('%H:%M:%S')}] "
                      f"{done_count}/{total_count} chunks done | "
                      f"Batches: {total_batches_done}/{total_batches_all} ({overall_pct:.1f}%) | "
                      f"{format_elapsed(elapsed_total)} elapsed | {eta_str}")
                # Per-chunk detail (2 per line for compactness)
                for i in range(0, len(chunk_details), 4):
                    row = "  ".join(chunk_details[i:i+4])
                    print(f"      {row}")

            if active_procs:
                time.sleep(2)

        pool_elapsed = time.time() - pool_start

        # Check for failures
        failed_chunks = [i for i, (ok, _) in results.items() if not ok]
        if failed_chunks:
            print(f"\n  FAILED chunks: {failed_chunks}")
            print(f"  Check logs in: {chunks_dir}")
            print(f"  Re-run this cell to retry — completed chunks will be skipped.")
            raise RuntimeError(
                f"Chunks failed for {name}: {failed_chunks}. "
                f"Re-run this cell to resume."
            )

        print(f"  All chunks done in {pool_elapsed:.1f}s ({pool_elapsed/60:.1f}m)")

    # ── STEP 3: MERGE ────────────────────────────────────────────────────
    print(f"  Merging {chunks_meta['num_chunks']} chunks...")
    merged_dfs = []
    for chunk_meta in chunks_meta["chunks"]:
        chunk_out = chunks_dir / chunk_meta["chunk_output"]
        chunk_df = pd.read_csv(chunk_out)
        merged_dfs.append(chunk_df)

    merged = pd.concat(merged_dfs, ignore_index=True)
    config.perplexity_raw.mkdir(parents=True, exist_ok=True)
    merged.to_csv(raw_out, index=False)
    print(f"  Merged: {len(merged):,} rows -> {raw_out}")

    if len(merged) != total_rows:
        print(f"  WARNING: Expected {total_rows:,} rows but got {len(merged):,}")

    # Update metadata with merged status
    chunks_meta["status"] = "merged"
    chunks_meta["merged_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
    chunks_meta["merged_output"] = str(raw_out)
    chunks_meta["merged_rows"] = len(merged)
    save_chunks_metadata(chunks_dir, chunks_meta)

    # Clear GPU memory
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    # ── STEP 4: FEATURE CONVERSION ───────────────────────────────────────
    feat_dir = config.get_features_dir(name)
    df_numeric, feat_out = convert_to_features(pd.read_csv(raw_out), feat_dir)
    if df_numeric is not None:
        print(f"  Features: {len(df_numeric):,} rows x {len(df_numeric.columns)} cols -> {feat_out}")
    else:
        print(f"  Features already converted: {feat_out.name}")

print("\n" + "=" * 70)
print("Parallel chunk extraction complete.")
print("=" * 70)


PARALLEL CHUNK EXTRACTION (cell 21c)
  Chunks per dataset:     10
  Max parallel workers:   10
  Mixed precision (FP16): disabled
  torch.compile:          disabled
  Batch size:             64
  Device:                 cuda

DATASET: norm-EFCAMDAT-train
  No existing chunks — creating fresh split
  Total rows: 79,998
  Chunks: 10 x ~8,000 rows
  Chunk CSVs saved to: /content/drive/MyDrive/_p/cefr-classification/checkpoints/gpt2-native-20260313-192642/norm-EFCAMDAT-train-20260313-192642/chunks
  Processing 10 chunks with 10 workers...
    [15:25:28] LAUNCHED chunk 00 (8,000 rows, PID 13945)
    [15:25:28] LAUNCHED chunk 01 (8,000 rows, PID 13946)
    [15:25:28] LAUNCHED chunk 02 (8,000 rows, PID 13947)
    [15:25:28] LAUNCHED chunk 03 (8,000 rows, PID 13949)
    [15:25:28] LAUNCHED chunk 04 (8,000 rows, PID 13950)
    [15:25:29] LAUNCHED chunk 05 (8,000 rows, PID 13952)
    [15:25:29] LAUNCHED chunk 06 (8,000 rows, PID 13953)
    [15:25:29] LAUNCHED chunk 07 (8,000 rows, PID 13956)
   

In [ ]:
%%time
import subprocess
import sys
import pandas as pd

# Initialize global model_names storage
model_names = {}

features_file = config.get_features_dir("norm-EFCAMDAT-train") / "features_dense.csv"
labels_csv = config.experiment_base / "ml-training-data" / "norm-EFCAMDAT-train.csv"

# Check if labels need trimming (due to --limit)
n_features = len(pd.read_csv(features_file))
df_labels = pd.read_csv(labels_csv)
if len(df_labels) != n_features:
    print(f"  Trimming labels from {len(df_labels)} to {n_features} rows")
    trimmed = config.trimmed_labels / "norm-EFCAMDAT-train.csv"
    df_labels.head(n_features).to_csv(trimmed, index=False)
    labels_csv = trimmed

for clf_type in config.classifiers:
    model_name = f"norm-EFCAMDAT-train_{clf_type}_gpt2native"
    model_names[clf_type] = model_name
    model_dir = config.experiment_base / "feature-models" / "classifiers" / model_name

    if (model_dir / "classifier.pkl").exists():
        print(f"  Already trained: {model_name}")
        continue

    print(f"\n  Training {clf_type}...")
    cmd = [
        sys.executable, "-m", "src.train_classifiers",
        "-e", str(config.experiment_base),
        "--features-file", str(features_file),
        "--labels-csv", str(labels_csv),
        "--cefr-column", config.cefr_col,
        "--classifier", clf_type,
        "--model-name", model_name,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(config.repo_dir))
    if result.returncode != 0:
        print(f"  STDERR: {result.stderr[-2000:]}")
        raise RuntimeError(f"Training failed for {clf_type}")
    for line in result.stdout.strip().split('\n')[-5:]:
        print(f"  {line}")

print(f"\nTrained models: {list(model_names.values())}")

---
## 5. Training & Evaluation

### Step 1: Train Classifiers

%%time
import subprocess
import sys
import pandas as pd

test_sets = [n for n, role in config.datasets.items() if role == "test"]

for clf_type in config.classifiers:
    model_name = model_names[clf_type]
    for test_name in test_sets:
        results_dir = config.experiment_base / "results" / model_name / test_name

        if (results_dir / "evaluation_report.md").exists():
            print(f"  Already done: {clf_type} -> {test_name}")
            continue

        features_file = config.get_features_dir(test_name) / "features_dense.csv"
        labels_csv = config.experiment_base / "ml-test-data" / f"{test_name}.csv"

        # Check if labels need trimming
        n_features = len(pd.read_csv(features_file))
        df_labels = pd.read_csv(labels_csv)
        if len(df_labels) != n_features:
            print(f"  Trimming {test_name} labels: {len(df_labels)} -> {n_features}")
            trimmed = config.trimmed_labels / f"{test_name}.csv"
            df_labels.head(n_features).to_csv(trimmed, index=False)
            labels_csv = trimmed

        print(f"  Predicting: {clf_type} -> {test_name}")
        cmd = [
            sys.executable, "-m", "src.predict",
            "-e", str(config.experiment_base),
            "-m", model_name,
            "--features-file", str(features_file),
            "--labels-csv", str(labels_csv),
            "--cefr-column", config.cefr_col,
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(config.repo_dir))
        if result.returncode != 0:
            print(f"  STDERR: {result.stderr[-2000:]}")
            raise RuntimeError(f"Prediction failed: {model_name} -> {test_name}")
        for line in result.stdout.strip().split('\n')[-3:]:
            print(f"    {line}")

print("\nAll predictions complete.")

In [ ]:
%%time
import subprocess
import sys
import pandas as pd

test_sets = [n for n, role in config.datasets.items() if role == "test"]

for clf_type in config.classifiers:
    model_name = model_names[clf_type]
    for test_name in test_sets:
        results_dir = config.experiment_base / "results" / model_name / test_name

        if (results_dir / "evaluation_report.md").exists():
            print(f"  Already done: {clf_type} -> {test_name}")
            continue

        features_file = config.get_features_dir(test_name) / "features_dense.csv"
        labels_csv = config.experiment_base / "ml-test-data" / f"{test_name}.csv"

        # Check if labels need trimming
        n_features = len(pd.read_csv(features_file))
        df_labels = pd.read_csv(labels_csv)
        if len(df_labels) != n_features:
            print(f"  Trimming {test_name} labels: {len(df_labels)} -> {n_features}")
            trimmed = config.trimmed_labels / f"{test_name}.csv"
            df_labels.head(n_features).to_csv(trimmed, index=False)
            labels_csv = trimmed

        print(f"  Predicting: {clf_type} -> {test_name}")
        cmd = [
            sys.executable, "-m", "src.predict",
            "-e", str(config.experiment_base),
            "-m", model_name,
            "--features-file", str(features_file),
            "--labels-csv", str(labels_csv),
            "--cefr-column", config.cefr_col,
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(config.repo_dir))
        if result.returncode != 0:
            print(f"  STDERR: {result.stderr[-2000:]}")
            raise RuntimeError(f"Prediction failed: {model_name} -> {test_name}")
        for line in result.stdout.strip().split('\n')[-3:]:
            print(f"    {line}")

print("\nAll predictions complete.")

import subprocess
import sys

summary_path = config.experiment_base / "results_summary.md"

cmd = [
    sys.executable, "-m", "src.report",
    "-e", str(config.experiment_base),
    "--rank", "accuracy",
    "--summary-report", str(summary_path),
    "--include-all-datasets",
    "-v",
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(config.repo_dir))
if result.returncode != 0:
    print(f"STDERR: {result.stderr[-2000:]}")
    raise RuntimeError("Report generation failed")
print(result.stdout[-1000:])

In [ ]:
---
## 9. Results

Display the results summary.

from IPython.display import Markdown, display

summary_path = config.experiment_base / "results_summary.md"

if summary_path.exists():
    display(Markdown(summary_path.read_text()))
else:
    print("No results summary found. Run Step 5 above.")

In [ ]:
### Individual Evaluation Reports

View detailed per-model, per-dataset evaluation reports.

---
## 10. Download Results (Optional)

Download the experiment outputs to your local machine.

!cd /content && zip -r -q experiment-results.zip results/ results_summary.md

from google.colab import files
files.download('/content/experiment-results.zip')

In [ ]:
# Zip the experiment directory for download
!cd /content && zip -r -q experiment-results.zip experiment/results/ experiment/results_summary.md

from google.colab import files
files.download('/content/experiment-results.zip')

In [ ]:
%%time
import subprocess
import sys
import time
from pathlib import Path
from collections import defaultdict

print("="*80)
print("PARALLEL DATASET EXTRACTION - GPU OPTIMIZATION")
print("="*80)

# ── CONFIGURATION ────────────────────────────────────────────────────────────
BATCH_SIZE_OPTIMIZED = 256  # Increase from 64 to 256 for 2-3x speedup
MAX_PARALLEL_PROCESSES = 2  # Run 2 datasets simultaneously (adjust: 2, 3, or 4)
DATASETS_TO_PROCESS = list(config.datasets.keys())  # Process all 4 datasets

print(f"\nConfiguration:")
print(f"  Batch Size: {BATCH_SIZE_OPTIMIZED}")
print(f"  Max Parallel: {MAX_PARALLEL_PROCESSES}")
print(f"  Datasets: {DATASETS_TO_PROCESS}")
print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"  Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── PARALLEL PROCESSING ─────────────────────────────────────────────────────
print(f"\n{'='*80}")
print("LAUNCHING PARALLEL EXTRACTION")
print(f"{'='*80}\n")

processes = {}  # {dataset_name: subprocess.Popen}
completed = {}  # {dataset_name: (success, time_elapsed)}
start_time = time.time()

# Launch datasets in batches (MAX_PARALLEL_PROCESSES at a time)
dataset_queue = DATASETS_TO_PROCESS.copy()
active_processes = {}

while dataset_queue or active_processes:
    # Launch new processes while under limit
    while len(active_processes) < MAX_PARALLEL_PROCESSES and dataset_queue:
        dataset_name = dataset_queue.pop(0)

        checkpoint_dir = config.get_checkpoint_dir(dataset_name)
        src_csv = config.data_path / f"{dataset_name}.csv"
        raw_out = config.perplexity_raw / f"{dataset_name}.csv"
        limit = config.limits.get(dataset_name)

        # Check if already done
        if raw_out.exists():
            print(f"✓ {dataset_name}: Already extracted, skipping")
            completed[dataset_name] = (True, 0)
            continue

        # Build command
        cmd = [
            sys.executable, "-m", "src.extract_perplexity_features_batched",
            "-i", str(src_csv),
            "--text-column", "text",
            "-m", "gpt2",
            "-d", DEVICE,
            "--aggregate-only",
            "--batch-size", str(BATCH_SIZE_OPTIMIZED),
            "--checkpoint-dir", str(checkpoint_dir),
            "--long-text-strategy", config.long_text_strategy,
            "-f", "csv",
            "-o", str(raw_out),
        ]

        # Add resume if checkpoint exists
        if checkpoint_dir.exists():
            cmd.append("--resume")

        if limit is not None:
            cmd.extend(["--limit", str(limit)])

        # Launch process
        try:
            p = subprocess.Popen(
                cmd,
                cwd=str(config.repo_dir),
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )
            active_processes[dataset_name] = {
                "process": p,
                "start_time": time.time(),
                "src_csv": src_csv,
            }
            print(f"▶ [{time.strftime('%H:%M:%S')}] LAUNCHED: {dataset_name} (PID: {p.pid})")
        except Exception as e:
            print(f"✗ {dataset_name}: Failed to launch - {e}")
            completed[dataset_name] = (False, 0)

    # Check for completed processes
    finished = []
    for dataset_name, proc_info in active_processes.items():
        p = proc_info["process"]
        if p.poll() is not None:  # Process finished
            elapsed = time.time() - proc_info["start_time"]

            if p.returncode == 0:
                print(f"✓ [{time.strftime('%H:%M:%S')}] DONE: {dataset_name} ({elapsed:.1f}s)")
                completed[dataset_name] = (True, elapsed)
            else:
                stdout, stderr = p.communicate()
                print(f"✗ [{time.strftime('%H:%M:%S')}] FAILED: {dataset_name}")
                print(f"   Error: {stderr[-500:] if stderr else 'Unknown error'}")
                completed[dataset_name] = (False, elapsed)

            finished.append(dataset_name)

    # Remove finished processes
    for dataset_name in finished:
        del active_processes[dataset_name]

    # Sleep to avoid busy-waiting
    if active_processes:
        time.sleep(2)

# ── SUMMARY ──────────────────────────────────────────────────────────────────
total_time = time.time() - start_time
successful = sum(1 for success, _ in completed.values() if success)
failed = len(completed) - successful

print(f"\n{'='*80}")
print("PARALLEL EXTRACTION SUMMARY")
print(f"{'='*80}")
print(f"Total Time: {total_time:.1f}s ({total_time/60:.1f}m)")
print(f"Successful: {successful}/{len(completed)}")
print(f"Failed: {failed}/{len(completed)}")
print(f"\nTimings:")
for dataset_name in DATASETS_TO_PROCESS:
    if dataset_name in completed:
        success, elapsed = completed[dataset_name]
        status = "✓" if success else "✗"
        print(f"  {status} {dataset_name:25s}: {elapsed:6.1f}s")

print(f"\n{'='*80}")
print(f"Speedup vs Sequential: ~{MAX_PARALLEL_PROCESSES}x (theoretical: {MAX_PARALLEL_PROCESSES}x / overhead)")
print(f"GPU Batch Size: {BATCH_SIZE_OPTIMIZED} (increased from 64)")
print(f"{'='*80}")

In [ ]:
import subprocess
import sys
import time
import threading
from pathlib import Path

print("="*70)
print("GPU OPTIMIZATION TEST CELL")
print("="*70)

# ── GPU MEMORY ANALYSIS ─────────────────────────────────────────────────
print("\n1. GPU MEMORY ANALYSIS")
print("-" * 70)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Estimate memory per batch
print("\nMemory Usage Estimates for GPT-2 (estimated):")
batch_sizes = [32, 64, 128, 256, 512]
for bs in batch_sizes:
    # Rough estimate: ~100-150MB per batch_size for GPT-2
    estimated_mb = (bs * 0.15)  # 150MB per batch_size
    estimated_gb = estimated_mb / 1024
    status = "✓ SAFE" if estimated_gb < 12 else "⚠ RISKY" if estimated_gb < 14.5 else "✗ OOM LIKELY"
    print(f"  batch_size={bs:3d}: ~{estimated_gb:5.2f} GB  {status}")

# ── STRATEGY 1: BATCH SIZE COMPARISON ───────────────────────────────────
print("\n\n2. STRATEGY 1: INCREASE BATCH SIZE")
print("-" * 70)
print("To use this strategy:")
print("  1. Update config in cell 4:")
print("       config = ExperimentConfig(")
print("           batch_size=128,  # Change from 64 to 128")
print("           ...")
print("       )")
print("  2. Re-run extraction cell (cell 21)")
print("")
print("Speedup estimate:")
print("  - batch_size=64→128: ~1.4-1.6x faster")
print("  - batch_size=64→256: ~1.8-2.0x faster (if no OOM)")

# ── STRATEGY 2: PARALLEL DATASET EXTRACTION ─────────────────────────────
print("\n\n3. STRATEGY 2: PARALLEL DATASET PROCESSING")
print("-" * 70)
print("Run multiple datasets simultaneously on GPU")
print("")

# Show the parallel processing code template
parallel_code = """
# TEMPLATE: Parallel dataset processing
import subprocess
import threading

processes = []
datasets_to_process = ["norm-EFCAMDAT-train", "norm-EFCAMDAT-test", "norm-CELVA-SP"]

for dataset_name in datasets_to_process:
    checkpoint_dir = config.get_checkpoint_dir(dataset_name)
    src_csv = config.data_path / f"{dataset_name}.csv"
    raw_out = config.perplexity_raw / f"{dataset_name}.csv"

    cmd = [
        sys.executable, "-m", "src.extract_perplexity_features_batched",
        "-i", str(src_csv),
        "--text-column", "text",
        "-m", "gpt2",
        "-d", "cuda",
        "--aggregate-only",
        "--batch-size", "64",
        "--checkpoint-dir", str(checkpoint_dir),
        "--long-text-strategy", "immediate_context",
        "-f", "csv",
        "-o", str(raw_out),
        "--resume",
    ]

    # Launch process asynchronously
    p = subprocess.Popen(cmd, cwd=str(config.repo_dir))
    processes.append((dataset_name, p))
    print(f"  Launched: {dataset_name} (PID: {p.pid})")

# Wait for all to complete
print("\\nWaiting for all datasets to complete...")
for dataset_name, p in processes:
    returncode = p.wait()
    status = "✓ DONE" if returncode == 0 else "✗ FAILED"
    print(f"  {dataset_name}: {status}")
"""

print(f"Code template:\n{parallel_code}")

# ── HYBRID STRATEGY: BATCH SIZE + PARALLEL ──────────────────────────────
print("\n4. HYBRID STRATEGY (RECOMMENDED): Batch Size + Parallel")
print("-" * 70)
print("Combine both approaches for maximum GPU utilization:")
print("")
print("Step 1: Update config in cell 4:")
print("  batch_size=128  # Increased from 64")
print("")
print("Step 2: Use parallel processing template above with batch_size=128")
print("")
print("Expected Results:")
print("  - Individual: batch_size=64 → ~8 GB usage, moderate speed")
print("  - Batch size 128: → ~12-14 GB usage, ~1.5x faster")
print("  - Parallel x2 (batch=128): → ~14 GB sustained, ~2.8-3.0x total speedup")
print("  - Parallel x3 (batch=128): → Limited by GPU memory, ~3.5-4.0x possible")

# ── GPU UTILIZATION MONITORING ──────────────────────────────────────────
print("\n\n5. MONITOR GPU USAGE (run in another cell during processing):")
print("-" * 70)
monitoring_code = """
# Run this in a separate cell to monitor GPU usage
!watch -n 1 nvidia-smi
# Or for continuous output:
!nvidia-smi -l 1
"""
print(monitoring_code)

print("\n" + "="*70)
print("END GPU OPTIMIZATION TEST")
print("="*70)

---
## 11. GPU OPTIMIZATION & PARALLEL EXECUTION (Testing)

Two strategies to drastically increase GPU utilization:

### Strategy 1: Increase Batch Size
- **Current:** batch_size=64 (typically 40-60% GPU usage on T4)
- **Goal:** batch_size=128 or 256 for 80-95% GPU usage
- **Trade-off:** Higher memory usage, faster processing

**GPU Memory Guidelines (T4 = 15.6 GB):**
- batch_size=64: ~8 GB (safe)
- batch_size=128: ~12-14 GB (optimal)
- batch_size=256: ~15 GB (risky - may OOM)
- batch_size=512: ~30 GB (requires A100)

### Strategy 2: Parallel Dataset Processing
- Run multiple datasets simultaneously on GPU
- Uses `subprocess.Popen()` for async execution
- Maximizes GPU pipeline parallelism
- Can combine with Strategy 1

**Expected Improvement:**
- Serial (baseline): 100% utilization = 1x speed
- Parallel x2: ~180-190% utilization = 1.8-1.9x speed
- Parallel x4: ~190-195% utilization (GPU bottleneck)